# Kaggle v6 零重叠空间隔离诊断训练

用于 `dataset_v6_spatial811`：Train/Val/Test = 494/63/55，512×512 tile，三个 split 之间无原始像素重叠。

默认运行 LTL-Net、ResNet50、seed=42、80 epochs、无新增数据增强。首跑只改变数据协议，用来量化40%重叠随机划分与零重叠空间隔离的差异。


In [ ]:
import os
import sys
import csv
import json
import subprocess
import importlib.util
import importlib.metadata
from collections import defaultdict
from pathlib import Path

REPO_URL = 'https://github.com/song110585-cpu/lunar-linear.git'
REPO_BRANCH = 'test-new-module'
REPO_DIR = Path('/kaggle/working/lunar-linear')
PROJECT_DIR = REPO_DIR / 'LTL-Net'
OUTPUT_ROOT = Path('/kaggle/working')

DATA_CANDIDATES = [
    Path('/kaggle/input/datasets/yuanssy/v6data/dataset_v6_spatial811'),
    Path('/kaggle/input/datasets/changyasong/v6data/dataset_v6_spatial811'),
    Path('/kaggle/input/v6data/dataset_v6_spatial811'),
]

print('Python:', sys.version)
print('Kaggle input roots:')
for path in Path('/kaggle/input').glob('*'):
    print(' -', path)


## 环境检查


In [ ]:
required = [('rasterio', 'rasterio'), ('matplotlib', 'matplotlib'), ('tqdm', 'tqdm')]
missing = [package for module, package in required if importlib.util.find_spec(module) is None]
try:
    smp_version = importlib.metadata.version('segmentation-models-pytorch')
except importlib.metadata.PackageNotFoundError:
    smp_version = None
if importlib.util.find_spec('segmentation_models_pytorch') is None or smp_version != '0.5.0':
    missing.append('segmentation-models-pytorch==0.5.0')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import torch
import numpy as np
import rasterio
import segmentation_models_pytorch as smp

assert torch.cuda.is_available(), 'Kaggle没有开启GPU，请在Notebook settings中选择GPU。'
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU capability:', torch.cuda.get_device_capability(0))
print('Torch CUDA arch list:', torch.cuda.get_arch_list())
print('SMP:', smp.__version__)
print('Rasterio:', rasterio.__version__)
x = torch.ones(16, device='cuda')
print('CUDA smoke:', float((x + 1).mean()))
del x
torch.cuda.empty_cache()


## 获取并锁定代码版本


In [ ]:
if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)])
elif not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'目录存在但不是Git仓库: {REPO_DIR}')
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])

assert (PROJECT_DIR / 'scripts' / 'train_ltl.py').is_file(), PROJECT_DIR
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
branch = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--abbrev-ref', 'HEAD'], text=True).strip()
print('Project:', PROJECT_DIR)
print('Git branch:', branch)
print('Git commit:', commit)


## 数据版本与零重叠核验


In [ ]:
EXPECTED_TILES = {'train': 494, 'val': 63, 'test': 55}
EXPECTED_MEAN = np.array([
    0.14634284944967138, 0.6073079654785911, 0.19560334930002798,
    0.5146424734578662, 0.4709976669495292,
])
EXPECTED_STD = np.array([
    0.06626422267714538, 0.34714253416282376, 0.2144817435820576,
    0.15443829274986642, 0.15999780505000355,
])

def tif_files(folder):
    return sorted([*folder.glob('*.tif'), *folder.glob('*.tiff')])

existing = [path for path in DATA_CANDIDATES if path.is_dir()]
if not existing:
    discovered = [p for p in Path('/kaggle/input').rglob('dataset_v6_spatial811') if p.is_dir()]
    existing.extend(discovered)
assert existing, '没有找到dataset_v6_spatial811，请确认已Add data。'
DATA_ROOT = existing[0]
print('Dataset:', DATA_ROOT)

for split, expected in EXPECTED_TILES.items():
    images = tif_files(DATA_ROOT / split / 'image')
    masks = tif_files(DATA_ROOT / split / 'mask')
    assert len(images) == expected, f'{split} image={len(images)}, expected={expected}'
    assert len(masks) == expected, f'{split} mask={len(masks)}, expected={expected}'
    assert {p.stem for p in images} == {p.stem for p in masks}, f'{split} image-mask mismatch'
    mask_by_stem = {p.stem: p for p in masks}
    for index in sorted({0, len(images) // 2, len(images) - 1}):
        with rasterio.open(images[index]) as src:
            image = src.read()
        with rasterio.open(mask_by_stem[images[index].stem]) as src:
            mask = src.read(1)
        assert image.shape == (5, 512, 512), (images[index], image.shape)
        assert mask.shape == (512, 512), (mask_by_stem[images[index].stem], mask.shape)
        bad = ~np.isfinite(image) | (image < -1e10)
        invalid_ratio = float(np.any(bad, axis=0).mean())
        assert invalid_ratio <= 0.05, f'invalid ratio too high: {images[index]}, {invalid_ratio:.4%}'
        assert set(np.unique(mask)).issubset({0, 1, 2, 3, 4}), np.unique(mask)
    print(f'{split}: {len(images)} pairs OK')

stats_path = DATA_ROOT / 'normalization_stats.json'
assert stats_path.is_file(), stats_path
stats = json.loads(stats_path.read_text(encoding='utf-8'))
assert np.allclose(stats['mean'], EXPECTED_MEAN, rtol=0, atol=1e-12), stats['mean']
assert np.allclose(stats['std'], EXPECTED_STD, rtol=0, atol=1e-12), stats['std']

manifest_path = DATA_ROOT / 'tile_manifest.csv'
assert manifest_path.is_file(), manifest_path
with manifest_path.open('r', encoding='utf-8-sig', newline='') as handle:
    manifest = list(csv.DictReader(handle))
assert len(manifest) == sum(EXPECTED_TILES.values()), len(manifest)
windows = [(r['asset_id'], int(r['row']), int(r['col'])) for r in manifest]
assert len(windows) == len(set(windows)), '存在完全重复窗口'
group_splits = defaultdict(set)
for row in manifest:
    group_splits[row['group_id']].add(row['split'])
crossing_groups = [group for group, splits in group_splits.items() if len(splits) > 1]
assert not crossing_groups, f'空间组跨split: {crossing_groups[:5]}'

by_asset = defaultdict(list)
for row in manifest:
    by_asset[row['asset_id']].append((int(row['row']), int(row['col']), row['split']))
overlap_pairs = []
for asset_id, items in by_asset.items():
    for i, (r1, c1, s1) in enumerate(items):
        for r2, c2, s2 in items[i + 1:]:
            if s1 != s2 and abs(r1 - r2) < 512 and abs(c1 - c2) < 512:
                overlap_pairs.append((asset_id, r1, c1, s1, r2, c2, s2))
assert not overlap_pairs, f'发现跨split像素重叠: {overlap_pairs[:3]}'
print('Dataset verification PASS: 494/63/55, unique windows, no cross-split pixel overlap')
print(json.dumps(stats, ensure_ascii=False, indent=2))


## 训练配置

首跑保持 `AUGMENTATION='none'`。若之后增加增强，必须使用新RUN_SUFFIX并作为独立实验记录。


In [ ]:
MODEL_KIND = 'ltl'          # 'ltl' 或 'baseline'
BASELINE_MODEL = 'DeepLabV3Plus'
ENCODER = 'resnet50'
SEED = 42
EPOCHS = 80
MAX_STEPS = 0              # 正式训练必须为0
NUM_WORKERS = 2
AUGMENTATION = 'none'      # 首跑固定none，不在notebook临时改变训练数据
RUN_SUFFIX = 'noaug_formal80'

assert MODEL_KIND in {'ltl', 'baseline'}
assert MAX_STEPS == 0, '正式诊断不能限制batch数'
assert AUGMENTATION == 'none', '当前训练脚本未启用增强；首跑应保持none'
model_name = 'LTLNet' if MODEL_KIND == 'ltl' else BASELINE_MODEL
run_name = f'v6_zerooverlap_{model_name}_seed{SEED}_{RUN_SUFFIX}'
result_dir = OUTPUT_ROOT / f'result_{run_name}'
assert not result_dir.exists(), f'结果目录已存在，为防止覆盖请修改RUN_SUFFIX: {result_dir}'

common = [
    '--encoder', ENCODER, '--data-dir', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_ROOT), '--seed', str(SEED),
    '--epochs', str(EPOCHS), '--max-steps', str(MAX_STEPS),
    '--num-workers', str(NUM_WORKERS), '--run-name', run_name,
]
if MODEL_KIND == 'ltl':
    command = [sys.executable, str(PROJECT_DIR / 'scripts' / 'train_ltl.py'), *common]
else:
    command = [sys.executable, str(PROJECT_DIR / 'scripts' / 'train_baseline.py'), '--model', BASELINE_MODEL, *common]

run_config = {
    'model_kind': MODEL_KIND, 'model_name': model_name, 'encoder': ENCODER,
    'seed': SEED, 'epochs': EPOCHS, 'augmentation': AUGMENTATION,
    'dataset': str(DATA_ROOT), 'git_commit': commit, 'command': command,
}
config_path = OUTPUT_ROOT / f'config_{run_name}.json'
config_path.write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding='utf-8')
print('Command:')
print(' '.join(command))
print('Config:', config_path)
print('Result:', result_dir)


## 开始正式训练


In [ ]:
subprocess.check_call(command, cwd=PROJECT_DIR)


## 结果核验与统一评估


In [ ]:
metrics_path = result_dir / 'metrics.json'
checkpoint_path = result_dir / 'best_model.pth'
assert metrics_path.is_file(), metrics_path
assert checkpoint_path.is_file(), checkpoint_path
result = json.loads(metrics_path.read_text(encoding='utf-8'))
print(json.dumps(result, ensure_ascii=False, indent=2))

evaluation_dir = result_dir / 'evaluation'
eval_command = [
    sys.executable, str(PROJECT_DIR / 'scripts' / 'evaluate_segmentation.py'),
    '--model', model_name, '--encoder', ENCODER, '--data-dir', str(DATA_ROOT),
    '--checkpoint', str(checkpoint_path), '--output-dir', str(evaluation_dir),
    '--num-workers', str(NUM_WORKERS), '--samples-per-group', '4',
]
print('Evaluation command:')
print(' '.join(eval_command))
subprocess.check_call(eval_command, cwd=PROJECT_DIR)


## 打包下载


In [ ]:
import shutil
shutil.copy2(config_path, result_dir / config_path.name)
archive_base = OUTPUT_ROOT / result_dir.name
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=result_dir)
print('Download:', archive_path)
print('ZIP包含权重、指标、配置、训练历史、曲线、混淆矩阵和定性结果。')
